# Percepción de inseguridad y victimización en el DMQ: cambios entre las olas 2023-2024 y 2025-2026 de la EVPI

**Secretaría General de Seguridad Ciudadana y Gestión de Riesgos · Observatorio Metropolitano de Seguridad Ciudadana y Gestión de Riesgos (OMSCGR)**

Cuaderno de replicación del estudio *Percepción de inseguridad y victimización en el Distrito Metropolitano de Quito: cambios entre las olas 2023-2024 y 2025-2026 de la EVPI* (octubre de 2026). Reproduce todas las cifras, tablas y figuras del informe a partir de la base de la EVPI.

| Sección del cuaderno | Contenido del informe |
|---|---|
| 1-4. Preparación | Parámetros, base, covariables, 56 indicadores y funciones |
| 5. Muestra | Tabla 2 (encuestas por administración zonal) |
| 6. Emparejamiento y balance | Sección 4, Tabla 3 y Figura 1 |
| 7. Resultados principales | Sección 5, Tablas 1 y 4 a 10, Figuras 2 a 4 |
| 8. Robustez | Sección 7, Tabla 12 y Figura 6 |
| 9. Diferencias por grupo | Sección 6, Tabla 11 y Figura 5 |
| 10. Victimización y registros administrativos | Recuadro 3 |
| 11. Exportar | Excel con todas las tablas y carpeta con las figuras |

**Cómo usarlo:** ejecute las celdas en orden (*Entorno de ejecución → Ejecutar todas*) y cargue el archivo `.xlsx` de la EVPI cuando se le pida. La ejecución completa tarda unos minutos.

> Los resultados describen cambios entre olas de una encuesta; no son efectos causales y pueden reflejar cambios en el levantamiento además de cambios en las condiciones.

## 1. Parámetros y librerías
Todos los supuestos del análisis están en esta celda. Cambiarlos modifica los resultados respecto del informe.

In [ ]:
import os, io, zipfile, textwrap, warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

# --- Emparejamiento (especificación principal) ---
K_VECINOS = 3                                            # parejas de 2023-2024 por cada encuesta de 2025-2026
PESOS_DISTANCIA = {"ingreso": 3, "trabaja": 3, "edad_o": 2}  # peso extra en la distancia (resto = 1)
COL_PESO = "F_exp_personas"                              # factor de expansión de personas
COL_AZ = "ADMINISTRACION ZONAL"                          # emparejamiento exacto y control
COL_BARRIO = "dpa_barrio"                                # conglomerado de los errores y variante por barrio

# --- Robustez ---
PERCENTIL_RECORTE = 99                                   # recorte de pesos extremos
VARIANTES_ROBUSTEZ = ["k=1", "k=5", "Pesos recortados p99"]  # un cambio es robusto si sobrevive a estas tres

# --- Diferencias por grupo ---
INDICADORES_CLAVE = ["PSC_2", "PSC_14", "PSC_15", "PSC_16", "CC_5", "VI_PERS", "VI_6", "CIE_3_1"]

# --- Salidas ---
CARPETA = "salidas_estudio"
os.makedirs(CARPETA, exist_ok=True)

# Paleta institucional
NAVY, BLUE, CYAN = "#15206B", "#4E59AA", "#1D9EBE"
GREEN, ORANGE, LIME, AMBER, GREY = "#0FA27C", "#FF6840", "#80CB32", "#FFB500", "#94A3B8"
plt.rcParams["font.family"] = "DejaVu Sans"

## 2. Carga de la base
En Google Colab se abre el selector de archivos. Fuera de Colab, defina la ruta en `RUTA` o en la variable de entorno `EVPI_XLSX`.

In [ ]:
RUTA = os.environ.get("EVPI_XLSX", "261002_EVPI_2024_2026_VF_1.xlsx")
if not os.path.exists(RUTA):
    try:
        from google.colab import files
        subido = files.upload()
        RUTA = list(subido.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"No se encontró {RUTA}. Defina la ruta en RUTA o en EVPI_XLSX.")

df = pd.read_excel(RUTA).reset_index(drop=True)
print("Encuestas por ola:", df["AÑO"].value_counts().sort_index().to_dict())

def col(codigo):
    """Nombre completo de una columna a partir de su código (p. ej. 'PSC_1')."""
    return [c for c in df.columns if c.split(" - ")[0] == codigo][0]

## 3. Covariables y grupos
Las siete características socioeconómicas se usan en el emparejamiento y en el ajuste. Los faltantes de estudios e ingreso se imputan con la mediana. También se definen los grupos de la sección 9.

In [ ]:
df["w"] = df[COL_PESO].astype(float)
df["T"] = (df["AÑO"] == 2025).astype(float)               # 1 = ola 2025-2026; 0 = ola 2023-2024

df["mujer"] = (df[col("DD_1")] == "Mujer").astype(float)
df["edad_o"] = df["RANGO ETAREO"].map({"18 A 25": 1, "26 A 40": 2, "41 A 55": 3, "56 A 64": 4, "65 Y MÁS": 5}).astype(float)

orden_estudios = {"A": 0, "B": 2, "C": 1, "D": 4, "E": 3, "F": 6, "G": 5, "H": 8, "I": 7, "J": 9, "K": 9, "L": 9}
df["educ"] = df[col("DD_7")].str[0].map(orden_estudios).astype(float)
df["educ"] = df["educ"].fillna(df["educ"].median())

ing = df[col("DD_10")].str[0]
df["nsnc"] = (ing == "K").astype(float)                    # ingreso no reportado
df["ingreso"] = ing.map({c: i for i, c in enumerate("ABCDEFGHIJ")}).astype(float)
df["ingreso"] = df["ingreso"].fillna(df["ingreso"].median())

df["trabaja"] = df[col("DD_8")].str.startswith("1").astype(float)
df["nse3f"] = df["nse3"].astype(float)
COVS = ["mujer", "edad_o", "educ", "ingreso", "nsnc", "nse3f", "trabaja"]

# Grupos poblacionales
df["g_sexo"] = np.where(df["mujer"] == 1, "Mujeres", "Hombres")
df["g_edad"] = pd.cut(df["edad_o"], [0, 2, 4, 5], labels=["18 a 40 años", "41 a 64 años", "65 años y más"]).astype(str)
df["g_nse"] = df["nse3"].map({1: "NSE bajo", 2: "NSE medio", 3: "NSE alto"})
GRUPOS = {"Sexo": ("g_sexo", ["Hombres", "Mujeres"]),
          "Edad": ("g_edad", ["18 a 40 años", "41 a 64 años", "65 años y más"]),
          "Nivel socioeconómico": ("g_nse", ["NSE bajo", "NSE medio", "NSE alto"])}
df[COVS].describe().round(2)

## 4. Indicadores y funciones
Los 56 indicadores del tablero territorial y su dirección de mejora (`-1`: menos es mejor; `1`: más es mejor; `0`: sin dirección normativa). Se expresan en escala 0-100.

In [ ]:
# (id, bloque, nombre, dirección, universo, advertencia)
INDICADORES = [
    ('PSC_1', 'A. Percepción de inseguridad', 'Inseguridad alrededor de la casa', -1, 'Excluye "No aplica"', ''),
    ('PSC_2', 'A. Percepción de inseguridad', 'Inseguridad en el barrio', -1, 'Excluye "No aplica"', ''),
    ('PSC_4', 'A. Percepción de inseguridad', 'Inseguridad cerca de centros educativos', -1, 'Excluye "No aplica"', ''),
    ('PSC_5', 'A. Percepción de inseguridad', 'Inseguridad en mercados y ferias', -1, 'Excluye "No aplica"', ''),
    ('PSC_6', 'A. Percepción de inseguridad', 'Inseguridad en cajeros de la vía pública', -1, 'Excluye "No aplica"', ''),
    ('PSC_7', 'A. Percepción de inseguridad', 'Inseguridad en espacios públicos de la ciudad', -1, 'Excluye "No aplica"', ''),
    ('PSC_8', 'A. Percepción de inseguridad', 'Inseguridad en el Metro de Quito', -1, 'Excluye "No aplica"', ''),
    ('PSC_9', 'A. Percepción de inseguridad', 'Inseguridad en trole, Ecovía y corredores', -1, 'Excluye "No aplica"', ''),
    ('PSC_11', 'A. Percepción de inseguridad', 'Inseguridad en taxis amarillos', -1, 'Excluye "No aplica"', ''),
    ('PSC_12', 'A. Percepción de inseguridad', 'Inseguridad en transporte por aplicaciones', -1, 'Excluye "No aplica"', ''),
    ('PSC_13', 'A. Percepción de inseguridad', 'Inseguridad en espectáculos públicos', -1, 'Excluye "No aplica"', ''),
    ('PSC_14', 'A. Percepción de inseguridad', 'Inseguridad al caminar solo/a por el barrio de día', -1, 'Excluye "No aplica"', ''),
    ('PSC_15', 'A. Percepción de inseguridad', 'Inseguridad al caminar solo/a por el barrio de noche', -1, 'Excluye "No aplica"', ''),
    ('PSC_16', 'B. Evolución percibida de la delincuencia', 'Percepción de que la delincuencia aumentó en el barrio', -1, 'Excluye "No aplica"', ''),
    ('PSC_17', 'B. Evolución percibida de la delincuencia', 'Percepción de que la delincuencia aumentó en Quito', -1, 'Excluye "No aplica"', ''),
    ('PSC_18', 'B. Evolución percibida de la delincuencia', 'Percepción de que la delincuencia aumentó en el país', -1, 'Excluye "No aplica"', ''),
    ('CC_1', 'C. Problemas percibidos en el barrio', 'Peleas o riñas en el barrio', -1, 'Excluye Ns/Nc', ''),
    ('CC_2', 'C. Problemas percibidos en el barrio', 'Pandillas o grupos delictivos en el barrio', -1, 'Excluye Ns/Nc', ''),
    ('CC_3', 'C. Problemas percibidos en el barrio', 'Consumo de alcohol en espacio público', -1, 'Excluye Ns/Nc', ''),
    ('CC_4', 'C. Problemas percibidos en el barrio', 'Venta clandestina de alcohol', -1, 'Excluye Ns/Nc', ''),
    ('CC_5', 'C. Problemas percibidos en el barrio', 'Consumo de drogas en espacio público', -1, 'Excluye Ns/Nc', ''),
    ('CC_6', 'C. Problemas percibidos en el barrio', 'Venta de drogas en espacio público', -1, 'Excluye Ns/Nc', ''),
    ('CC_7', 'C. Problemas percibidos en el barrio', 'Calles sin iluminación', -1, 'Excluye Ns/Nc', ''),
    ('CC_8', 'C. Problemas percibidos en el barrio', 'Apropiación de viviendas por grupos delictivos', -1, 'Excluye Ns/Nc', ''),
    ('CC_9', 'C. Problemas percibidos en el barrio', 'Reclutamiento de niñas, niños y adolescentes', -1, 'Excluye Ns/Nc', ''),
    ('TA_3', 'C. Problemas percibidos en el barrio', 'Reporte de civiles armados en el barrio', -1, 'Excluye Ns/Nc', ''),
    ('VI_PERS', 'D. Victimización (12 meses)', 'Victimización personal (al menos un delito)', -1, 'Todos los encuestados', ''),
    ('VI_HOG', 'D. Victimización (12 meses)', 'Victimización del hogar (al menos un delito)', -1, 'Todos los hogares; ítems filtrados = 0 si no aplica', ''),
    ('VI_6', 'D. Victimización (12 meses)', 'Robo con violencia o amenaza', -1, 'Todos los encuestados', ''),
    ('VI_7', 'D. Victimización (12 meses)', 'Hurto', -1, 'Todos los encuestados', ''),
    ('VI_8', 'D. Victimización (12 meses)', 'Estafa', -1, 'Todos los encuestados', ''),
    ('VI_9', 'D. Victimización (12 meses)', 'Fraude bancario', -1, 'Todos los encuestados', ''),
    ('VI_11', 'D. Victimización (12 meses)', 'Agresión física por desconocido', -1, 'Todos los encuestados', ''),
    ('VI_12', 'D. Victimización (12 meses)', 'Amenaza creíble', -1, 'Todos los encuestados', ''),
    ('VI_13', 'D. Victimización (12 meses)', 'Extorsión', -1, 'Todos los encuestados', ''),
    ('VI_10', 'D. Victimización (12 meses)', 'Pago de soborno a funcionario', -1, 'Todos los encuestados', ''),
    ('VI_4', 'D. Victimización (12 meses)', 'Robo a la vivienda', -1, 'Todos los hogares', ''),
    ('VI_14', 'D. Victimización (12 meses)', 'Secuestro', -1, 'Todos los hogares', ''),
    ('VI_1_1', 'D. Victimización (12 meses)', 'Robo de vehículo', -1, 'Solo hogares a los que aplica el filtro', ''),
    ('VI_2_1', 'D. Victimización (12 meses)', 'Robo de moto', -1, 'Solo hogares a los que aplica el filtro', ''),
    ('VI_3', 'D. Victimización (12 meses)', 'Robo de autopartes', -1, 'Solo hogares a los que aplica el filtro', ''),
    ('VI_5_1', 'D. Victimización (12 meses)', 'Robo a negocio en la vivienda', -1, 'Solo hogares a los que aplica el filtro', ''),
    ('CIE_4_1', 'E. Confianza institucional', 'Confianza en la Policía Nacional', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CIE_5_1', 'E. Confianza institucional', 'Confianza en las Fuerzas Armadas', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CIE_3_1', 'E. Confianza institucional', 'Confianza en la Fiscalía', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CIE_1_1', 'E. Confianza institucional', 'Confianza en juzgados y cortes', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CIE_2_1', 'E. Confianza institucional', 'Confianza en el ECU 911', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CIM_4_1', 'E. Confianza institucional', 'Confianza en los Agentes de Control Metropolitano', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CIM_5_1', 'E. Confianza institucional', 'Confianza en la Secretaría de Seguridad', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CH_1', 'F. Cambios de hábitos por inseguridad', 'Reducción de salidas nocturnas por inseguridad', -1, 'Excluye Ns/Nc y No aplica', ''),
    ('CH_2', 'F. Cambios de hábitos por inseguridad', 'Reducción de visitas a espacios recreativos por inseguridad', -1, 'Excluye Ns/Nc y No aplica', ''),
    ('CH_3', 'F. Cambios de hábitos por inseguridad', 'Restricción de salidas de hijos menores por inseguridad', -1, 'Excluye Ns/Nc y No aplica', ''),
    ('CH_4', 'F. Cambios de hábitos por inseguridad', 'Reducción de visitas a lugares turísticos por inseguridad', -1, 'Excluye Ns/Nc y No aplica', ''),
    ('CH_5', 'F. Cambios de hábitos por inseguridad', 'Reducción de asistencia a eventos por inseguridad', -1, 'Excluye Ns/Nc y No aplica', ''),
    ('CH_6', 'F. Cambios de hábitos por inseguridad', 'Reducción de caminatas por el barrio por inseguridad', -1, 'Excluye Ns/Nc y No aplica', 'En el DMQ, la respuesta "No aplica" cae de 17% a 2% entre olas; base no plenamente comparable'),
    ('SP_5', 'F. Cambios de hábitos por inseguridad', 'Participación en iniciativas barriales de seguridad', 0, 'Todos los encuestados', ''),
]

In [ ]:
def cat(codigo):
    return df[col(codigo)].astype("string")

def marca(codigo, si=("1",), no=("2",)):
    """1 si la respuesta empieza con un código de 'si', 0 con uno de 'no'; NaN en el resto (No aplica, Ns/Nc)."""
    s = cat(codigo)
    y = pd.Series(np.nan, index=df.index)
    y[s.str.startswith(tuple(no)).fillna(False)] = 0.0
    y[s.str.startswith(tuple(si)).fillna(False)] = 1.0
    return y

def victimizacion(codigo):
    """Victimización con base 'todos': 1 si responde Sí, 0 en el resto."""
    s = cat(codigo)
    y = pd.Series(0.0, index=df.index)
    y[s.str.startswith("1").fillna(False)] = 1.0
    return y

Y = {}
for (i, bloque, nombre, direccion, base, adv) in INDICADORES:
    if i in ("PSC_16", "PSC_17", "PSC_18"):
        Y[i] = marca(i, si=("1",), no=("2", "3"))           # "aumentó" vs. se mantiene/disminuyó
    elif i.startswith("PSC_") or i.startswith("CC_") or i in ("TA_3", "SP_5"):
        Y[i] = marca(i)
    elif i.startswith("CH_"):                                # reducción de actividades por inseguridad
        s = cat(i)
        y = pd.Series(np.nan, index=df.index)
        y[s.str.contains("Aument|mantuv", regex=True).fillna(False)] = 0.0
        y[s.str.contains("Disminuy").fillna(False)] = 1.0
        Y[i] = y
    elif i.startswith(("CIE_", "CIM_")) or i in ("VI_1_1", "VI_2_1", "VI_3", "VI_5_1"):
        Y[i] = marca(i)                                      # confianza (entre quienes conocen) e ítems filtrados
    elif i.startswith("VI_") and i not in ("VI_PERS", "VI_HOG"):
        Y[i] = victimizacion(i)
Y["VI_PERS"] = pd.concat([Y[k].fillna(0) for k in ["VI_6", "VI_7", "VI_8", "VI_9", "VI_11", "VI_12", "VI_13"]], axis=1).max(axis=1)
Y["VI_HOG"] = pd.concat([Y[k].fillna(0) for k in ["VI_1_1", "VI_2_1", "VI_3", "VI_4", "VI_5_1", "VI_14"]], axis=1).max(axis=1)
YV = {k: v.values.astype(float) * 100 for k, v in Y.items()}
META = pd.DataFrame(INDICADORES, columns=["id", "bloque", "indicador", "dir", "universo", "advertencia"]).set_index("id")
print(len(YV), "indicadores construidos")

In [ ]:
T = df["T"].values
W = df["w"].values
BARRIO = df[COL_BARRIO].values
X_AJUSTE = np.column_stack([df[COVS].values.astype(float),
                            pd.get_dummies(df[COL_AZ], drop_first=True).astype(float).values])
_Z = (df[COVS] - df[COVS].mean()) / df[COVS].std()      # covariables estandarizadas (toda la base)

def emparejar(exactas, k=K_VECINOS, pesos_dist=PESOS_DISTANCIA, w=None, muestra=None):
    """Empareja cada encuesta 2025-2026 con sus k vecinas 2023-2024 dentro de las celdas 'exactas' (con reemplazo).
    Devuelve los pesos finales: el peso muestral para 2025-2026 y, para cada control, la suma de pesos/k de sus parejas."""
    w = W if w is None else w
    Z = _Z.copy()
    for v, p in pesos_dist.items():
        Z[v] = Z[v] * np.sqrt(p)
    Z = Z.values
    base = df if muestra is None else df[muestra]
    peso_control = np.zeros(len(df))
    for _, idx in base.groupby(exactas).groups.items():
        idx = np.array(list(idx))
        trat, ctrl = idx[T[idx] == 1], idx[T[idx] == 0]
        if len(trat) == 0 or len(ctrl) == 0:
            continue
        kk = min(k, len(ctrl))
        D = ((Z[trat][:, None, :] - Z[ctrl][None, :, :]) ** 2).sum(2)
        vecinos = np.argsort(D, axis=1)[:, :kk]
        for a, i in enumerate(trat):
            for j in vecinos[a]:
                peso_control[ctrl[j]] += w[i] / kk
    mw = np.where(T == 1, w, peso_control)
    if muestra is not None:
        mw[~np.asarray(muestra)] = 0
    return mw

def ols_cluster(y, X, w, grupos):
    """MCO ponderado con errores agrupados (CR1). Devuelve coeficientes, matriz de varianzas y número de conglomerados."""
    XtW = X.T * w
    A = XtW @ X
    b = np.linalg.lstsq(A, XtW @ y, rcond=None)[0]
    e = y - X @ b
    u, inv = np.unique(grupos, return_inverse=True)
    G = np.zeros((len(u), X.shape[1]))
    np.add.at(G, inv, X * (w * e)[:, None])
    g, n, k = len(u), len(y), X.shape[1]
    Ai = np.linalg.pinv(A)
    V = Ai @ (G.T @ G) @ Ai * g / (g - 1) * (n - 1) / (n - k)
    return b, V, g

def estimar(y, pesos, ajustar=True):
    """Diferencia 2025-2026 menos 2023-2024 (pp), con o sin ajuste por covariables y AZ; EE agrupados por barrio."""
    m = ~np.isnan(y) & (pesos > 0)
    X = np.column_stack([np.ones(m.sum()), T[m]] + ([X_AJUSTE[m]] if ajustar else []))
    b, V, g = ols_cluster(y[m], X, pesos[m], BARRIO[m])
    se = np.sqrt(V[1, 1])
    tt = T[m]
    return {"coef": b[1], "se": se, "p": 2 * (1 - stats.t.cdf(abs(b[1] / se), g - 1)),
            "media_2025": np.average(y[m][tt == 1], weights=pesos[m][tt == 1]),
            "media_2023": np.average(y[m][tt == 0], weights=pesos[m][tt == 0]),
            "n_2025": int((tt == 1).sum()), "n_2023": int((tt == 0).sum())}

def bh(p):
    """Valores q de Benjamini-Hochberg."""
    p = np.asarray(p); orden = np.argsort(p); q = np.empty(len(p)); prev = 1.0
    for r, i in enumerate(orden[::-1]):
        prev = min(prev, p[i] * len(p) / (len(p) - r)); q[i] = prev
    return q

def clasificar(coef, p, direccion):
    if direccion == 0:
        return "Sin dirección normativa" + (" (significativo)" if p < 0.05 else "")
    mejora = coef * direccion > 0
    if p < 0.05:
        return "Mejora significativa" if mejora else "Deterioro significativo"
    if p < 0.10:
        return "Señal de mejora por confirmar" if mejora else "Señal de deterioro por confirmar"
    return "Sin cambio significativo"

def fuente(ax_or_fig=None):
    return "Fuente: EVPI, olas 2023-2024 y 2025-2026. Elaboración: OMSCGR."

## 5. Muestra por administración zonal (Tabla 2)

In [ ]:
TABLA_MUESTRA = df.groupby([COL_AZ, "AÑO"]).size().unstack().rename(columns={2023: "2023-2024", 2025: "2025-2026"})
TABLA_MUESTRA["Total"] = TABLA_MUESTRA.sum(axis=1)
TABLA_MUESTRA.loc["Total DMQ"] = TABLA_MUESTRA.sum()
TABLA_MUESTRA

## 6. Emparejamiento principal y balance (sección 4, Tabla 3, Figura 1)
Diferencia estandarizada (DEE) entre olas antes y después del emparejamiento. Valores bajo 0,10 indican buen balance.

In [ ]:
MW = emparejar([COL_AZ])
print("Controles 2023-2024 usados:", int(((MW > 0) & (T == 0)).sum()), "de", int((T == 0).sum()))

filas = []
for v in COVS:
    t, c = df[T == 1], df[T == 0]
    sd = np.sqrt((t[v].var() + c[v].var()) / 2)
    m25 = np.average(t[v], weights=t["w"]); m23 = np.average(c[v], weights=c["w"]); m23e = np.average(c[v], weights=MW[T == 0])
    filas.append({"caracteristica": v, "media_2025": m25, "media_2023_sin_emparejar": m23, "media_2023_emparejado": m23e,
                  "dee_antes": (m25 - m23) / sd, "dee_despues": (m25 - m23e) / sd})
BALANCE = pd.DataFrame(filas)
display(BALANCE.round(2)) if "display" in globals() else print(BALANCE.round(2))

NOMBRES_COV = {"mujer": "Mujer", "edad_o": "Rango de edad", "educ": "Nivel de estudios", "ingreso": "Ingreso del hogar",
               "nsnc": "Ingreso no reportado", "nse3f": "Nivel socioeconómico", "trabaja": "Trabaja"}
fig, ax = plt.subplots(figsize=(6.7, 2.6), dpi=150)
b = BALANCE.iloc[::-1]; yy = np.arange(len(b))
ax.axvspan(-0.1, 0.1, color="#e9f4fb", zorder=0); ax.axvline(0, color="#64748b", lw=.8)
for k, r in enumerate(b.itertuples()):
    ax.plot([r.dee_antes, r.dee_despues], [k, k], color="#cbd5e1", lw=1.2)
ax.scatter(b.dee_antes, yy, color=ORANGE, s=22, label="Antes del emparejamiento", zorder=2)
ax.scatter(b.dee_despues, yy, color=GREEN, s=22, label="Después", zorder=3)
ax.set_yticks(yy); ax.set_yticklabels([NOMBRES_COV[v] for v in b.caracteristica], fontsize=7.5)
ax.set_xlabel("Diferencia estandarizada entre olas (banda: ±0,10)", fontsize=7.5); ax.tick_params(axis="x", labelsize=7)
[ax.spines[s].set_visible(False) for s in ("top", "right")]; ax.legend(fontsize=7, frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig(f"{CARPETA}/figura1_balance.png", dpi=220); plt.show()

## 7. Resultados principales (sección 5)
Para cada indicador: diferencia **simple** (sin emparejar), **emparejada** (sin ajuste) y **ajustada** (especificación principal: emparejamiento + regresión con covariables y AZ). Resultados en puntos porcentuales (pp).

In [ ]:
filas = []
for (i, bloque, nombre, direccion, base, adv) in INDICADORES:
    y = YV[i]
    simple = estimar(y, W, ajustar=False)
    emp = estimar(y, MW, ajustar=False)
    aj = estimar(y, MW, ajustar=True)
    filas.append({"id": i, "bloque": bloque, "indicador": nombre, "dir": direccion,
                  "media_2023": simple["media_2023"], "media_2025": simple["media_2025"],
                  "dif_simple": simple["coef"], "dif_emparejada": emp["coef"], "dif_ajustada": aj["coef"], "ee": aj["se"],
                  "ic95_inf": aj["coef"] - 1.96 * aj["se"], "ic95_sup": aj["coef"] + 1.96 * aj["se"], "p": aj["p"],
                  "n_2023": aj["n_2023"], "n_2025": aj["n_2025"]})
RES = pd.DataFrame(filas)
RES["q_BH"] = bh(RES["p"])
RES["clasificacion"] = [clasificar(r.dif_ajustada, r.p, r.dir) for r in RES.itertuples()]
print(RES["clasificacion"].value_counts().to_string())
print("q < 0,05:", int((RES.q_BH < 0.05).sum()), "| p < 0,05:", int((RES.p < 0.05).sum()))

### 7.1 Robustez de cada resultado
Se reestima con una pareja, cinco parejas y pesos recortados. **Robusto** = significativo (p < 0,05) y con el mismo signo en la especificación principal y en las tres variantes. Se agrega la variante con emparejamiento exacto por barrio (solo barrios presentes en ambas olas) como prueba de sensibilidad.

In [ ]:
w_rec = np.minimum(W, np.percentile(W, PERCENTIL_RECORTE))
barrios_ambas = set(df.loc[T == 0, COL_BARRIO]) & set(df.loc[T == 1, COL_BARRIO])
en_barrios_ambas = df[COL_BARRIO].isin(barrios_ambas).values
ESPEC = {"Principal (k=3)": MW,
         "k=1": emparejar([COL_AZ], k=1),
         "k=5": emparejar([COL_AZ], k=5),
         "Pesos recortados p99": emparejar([COL_AZ], w=w_rec),
         "Exacto por barrio": emparejar([COL_BARRIO], muestra=en_barrios_ambas)}
print(f"Barrios presentes en ambas olas: {len(barrios_ambas)} de {df[COL_BARRIO].nunique()}")

filas = []
for nombre_esp, pesos in ESPEC.items():
    for i in RES["id"]:
        r = estimar(YV[i], pesos, ajustar=True)
        filas.append({"especificacion": nombre_esp, "id": i, "coef": r["coef"], "se": r["se"], "p": r["p"]})
VARIANTES = pd.DataFrame(filas)
COEF = VARIANTES.pivot(index="id", columns="especificacion", values="coef")
PVAL = VARIANTES.pivot(index="id", columns="especificacion", values="p")

def es_robusto(i):
    signo = np.sign(COEF.loc[i, "Principal (k=3)"])
    return all(PVAL.loc[i, e] < 0.05 and np.sign(COEF.loc[i, e]) == signo for e in ["Principal (k=3)"] + VARIANTES_ROBUSTEZ)
RES["robusto"] = [es_robusto(i) if p < 0.05 else np.nan for i, p in zip(RES["id"], RES["p"])]
RES["dif_barrio"] = RES["id"].map(COEF["Exacto por barrio"])
RES["p_barrio"] = RES["id"].map(PVAL["Exacto por barrio"])
RES["clasificacion_barrio"] = [clasificar(c, p, d) for c, p, d in zip(RES.dif_barrio, RES.p_barrio, RES.dir)]

sig = RES[RES.p < 0.05]
print(f"Significativos: {len(sig)} | robustos: {int(sig.robusto.sum())}")
print("No robustos:", ", ".join(sig[sig.robusto == False].indicador))
print(f"Barrio vs. zona: mismo signo en {int((np.sign(RES.dif_ajustada) == np.sign(RES.dif_barrio)).sum())} de {len(RES)}; "
      f"correlación {np.corrcoef(RES.dif_ajustada, RES.dif_barrio)[0, 1]:.2f}; "
      f"clasificación igual en {int((RES.clasificacion == RES.clasificacion_barrio).sum())}")
print(f"Diferencia simple y ajustada con el mismo signo en {int((np.sign(RES.dif_simple) == np.sign(RES.dif_ajustada)).sum())} de {len(RES)}")

### 7.2 Tablas del informe
Tabla 1 (indicadores seleccionados), Tabla 4 (resumen por bloque), Tablas 5 a 10 (un bloque cada una) y Tabla 12 (matriz de robustez).

In [ ]:
def tabla_bloque(bloque):
    t = RES[RES.bloque == bloque][["indicador", "media_2023", "media_2025", "dif_ajustada", "ic95_inf", "ic95_sup", "p", "q_BH", "clasificacion", "robusto"]]
    return t.round({"media_2023": 1, "media_2025": 1, "dif_ajustada": 1, "ic95_inf": 1, "ic95_sup": 1, "p": 3, "q_BH": 3})

CLAVE_RESUMEN = ["PSC_2", "PSC_14", "PSC_15", "PSC_12", "PSC_16", "CC_5", "VI_PERS", "VI_6", "VI_7", "CIE_1_1", "CIE_3_1", "CH_6"]
TABLA1 = RES.set_index("id").loc[CLAVE_RESUMEN, ["indicador", "media_2023", "media_2025", "dif_ajustada", "clasificacion", "robusto"]].round(1)

def resumen_bloque(g):
    vc = g.clasificacion.value_counts()
    senal = vc.get("Señal de mejora por confirmar", 0) + vc.get("Señal de deterioro por confirmar", 0)
    mej, det = vc.get("Mejora significativa", 0), vc.get("Deterioro significativo", 0)
    return pd.Series({"indicadores": len(g), "mejora_signif": mej, "deterioro_signif": det, "senal_por_confirmar": senal,
                      "sin_cambio_o_sin_direccion": len(g) - mej - det - senal, "robustos": int(g.robusto.fillna(False).sum())})
TABLA4 = RES.groupby("bloque").apply(resumen_bloque)
TABLA12 = COEF.loc[sig["id"], ["Principal (k=3)", "k=1", "k=5", "Pesos recortados p99", "Exacto por barrio"]].round(1)
TABLA12.insert(0, "indicador", sig.set_index("id")["indicador"]); TABLA12["robusto"] = sig.set_index("id")["robusto"]

display(TABLA1) if "display" in globals() else print(TABLA1)
display(TABLA4) if "display" in globals() else print(TABLA4)
for b in RES.bloque.unique():
    print("\n" + b); print(tabla_bloque(b).to_string(index=False))

### 7.3 Figuras 2 a 4

In [ ]:
COLOR_CL = {"Mejora significativa": GREEN, "Deterioro significativo": ORANGE, "Señal de mejora por confirmar": LIME,
            "Señal de deterioro por confirmar": AMBER, "Sin cambio significativo": GREY}
color_de = lambda c: COLOR_CL.get(c, BLUE)

# Figura 2: indicadores por bloque según clasificación
fig, ax = plt.subplots(figsize=(6.7, 2.9), dpi=150)
tb = TABLA4.iloc[::-1]; izq = np.zeros(len(tb))
for clave, color, etiqueta in (("mejora_signif", GREEN, "Mejora significativa"), ("senal_por_confirmar", AMBER, "Señal por confirmar"),
                               ("sin_cambio_o_sin_direccion", "#CBD5E1", "Sin cambio / sin dirección"), ("deterioro_signif", ORANGE, "Deterioro significativo")):
    v = tb[clave].values.astype(float)
    ax.barh(tb.index, v, left=izq, color=color, label=etiqueta, height=.62, edgecolor="white", linewidth=.6)
    for k, (l, x) in enumerate(zip(izq, v)):
        if x > 0: ax.text(l + x / 2, k, str(int(x)), ha="center", va="center", fontsize=7, color="#1f2a44", fontweight="bold")
    izq += v
ax.set_xlabel("Número de indicadores", fontsize=8); ax.tick_params(labelsize=7.5)
[ax.spines[s].set_visible(False) for s in ("top", "right")]
ax.legend(fontsize=6.5, ncol=2, loc="upper center", bbox_to_anchor=(.42, -.22), frameon=False)
plt.tight_layout(); plt.savefig(f"{CARPETA}/figura2_bloques.png", dpi=220, bbox_inches="tight"); plt.show()

# Figuras 3 y 4: cambio ajustado con IC 95% (marcador hueco = significativo no robusto)
def forest(bloques, archivo, titulo):
    items = RES[RES.bloque.isin(bloques)]
    alto = 0.215 * (len(items) + 1.9 * len(bloques)) + 1.1
    fig, ax = plt.subplots(figsize=(6.7, alto), dpi=150)
    y, yt, yl, ultimo = 0, [], [], None
    for r in items.itertuples():
        if r.bloque != ultimo:
            y -= 0.9 if ultimo else 0
            ax.text(-0.02, y, r.bloque, transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=7.5, fontweight="bold", color=BLUE)
            y -= 1; ultimo = r.bloque
        c = color_de(r.clasificacion); sig_ = r.p < 0.05; hueco = sig_ and r.robusto is False
        ax.plot([r.ic95_inf, r.ic95_sup], [y, y], color=c, lw=1.6)
        ax.plot(r.dif_ajustada, y, "o", color="white" if hueco else c, mec=c if hueco else "white", mew=1.2 if hueco else .5, ms=4.6)
        yt.append(y); yl.append(r.indicador); y -= 1
    ax.axvline(0, color="#64748b", lw=.8); ax.set_yticks(yt); ax.set_yticklabels(yl, fontsize=6.4); ax.set_ylim(y + .3, .8)
    ax.set_xlabel("Diferencia ajustada 2025-2026 − 2023-2024 (pp), IC 95%", fontsize=7.5); ax.tick_params(axis="x", labelsize=7)
    [ax.spines[s].set_visible(False) for s in ("top", "right")]; ax.grid(axis="x", color="#e2e8f0", lw=.6); ax.set_axisbelow(True)
    h = [Line2D([0], [0], marker="o", color=c, lw=1.6, ms=4, label=l) for c, l in ((GREEN, "Mejora significativa"), (ORANGE, "Deterioro significativo"),
         (LIME, "Señal de mejora"), (AMBER, "Señal de deterioro"), (BLUE, "Sin dirección normativa"), (GREY, "Sin cambio significativo"))]
    h.append(Line2D([0], [0], marker="o", color=GREY, lw=0, ms=5, mfc="white", mec="#334155", label="Significativo no robusto"))
    ax.legend(handles=h, fontsize=6, ncol=4, loc="upper center", bbox_to_anchor=(.32, -.06), frameon=False)
    ax.set_title(titulo, fontsize=8, color=NAVY, loc="left")
    plt.tight_layout(); plt.savefig(f"{CARPETA}/{archivo}", dpi=220, bbox_inches="tight"); plt.show()

BLOQUES = list(RES.bloque.unique())
forest(BLOQUES[:3], "figura3_forest_percepcion.png", "Percepción, evolución percibida y problemas del barrio")
forest(BLOQUES[3:], "figura4_forest_victimizacion.png", "Victimización, confianza institucional y hábitos")

## 8. Robustez: zona frente a barrio (Figura 6)

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 4.2), dpi=150)
x, yv = RES.dif_ajustada.values, RES.dif_barrio.values
lim = max(abs(x).max(), abs(yv).max()) * 1.08
ax.plot([-lim, lim], [-lim, lim], color="#94a3b8", lw=.8, ls="--"); ax.axhline(0, color="#e2e8f0", lw=.6); ax.axvline(0, color="#e2e8f0", lw=.6)
ax.scatter(x, yv, s=14, color=BLUE, alpha=.85, edgecolors="white", linewidths=.4)
ax.set_xlabel("Emparejamiento por administración zonal (pp)", fontsize=7.5); ax.set_ylabel("Emparejamiento por barrio (pp)", fontsize=7.5)
ax.tick_params(labelsize=7); [ax.spines[s].set_visible(False) for s in ("top", "right")]; ax.set_aspect("equal")
plt.tight_layout(); plt.savefig(f"{CARPETA}/figura6_zona_vs_barrio.png", dpi=220); plt.show()

d = (RES.dif_barrio - RES.dif_ajustada).abs()
print(RES.loc[d.sort_values(ascending=False).index[:6], ["indicador", "dif_ajustada", "dif_barrio", "clasificacion", "clasificacion_barrio"]].round(1).to_string(index=False))

## 9. Diferencias por grupo poblacional (sección 6, Tabla 11, Figura 5)
Para cada dimensión (sexo, edad, nivel socioeconómico) se empareja de forma exacta por zona y grupo, y se estima el cambio de cada grupo en una sola regresión con interacciones. La **prueba de igualdad** (Wald) contrasta si el cambio es el mismo en todos los grupos.

In [ ]:
filas = []
for dimension, (col_g, niveles) in GRUPOS.items():
    mw_g = emparejar([COL_AZ, col_g])
    G = np.column_stack([(df[col_g] == l).astype(float).values for l in niveles])
    for i in INDICADORES_CLAVE:
        y = YV[i]; m = ~np.isnan(y) & (mw_g > 0)
        Gm, Tm = G[m], T[m]
        X = np.column_stack([Gm * Tm[:, None], Gm[:, 1:], X_AJUSTE[m], np.ones(m.sum())])
        b, V, g = ols_cluster(y[m], X, mw_g[m], BARRIO[m])
        L = len(niveles); R = np.zeros((L - 1, X.shape[1]))
        for r in range(L - 1):
            R[r, 0], R[r, r + 1] = 1, -1
        wald = (R @ b) @ np.linalg.solve(R @ V @ R.T, R @ b)
        p_igual = 1 - stats.chi2.cdf(wald, L - 1)
        for j, nivel in enumerate(niveles):
            se = np.sqrt(V[j, j])
            filas.append({"dimension": dimension, "grupo": nivel, "id": i, "indicador": META.loc[i, "indicador"],
                          "coef": b[j], "se": se, "ic95_inf": b[j] - 1.96 * se, "ic95_sup": b[j] + 1.96 * se,
                          "p": 2 * (1 - stats.t.cdf(abs(b[j] / se), g - 1)), "p_igualdad": p_igual,
                          "n_2025": int(((Tm == 1) & (Gm[:, j] == 1)).sum())})
SUBGRUPOS = pd.DataFrame(filas)
TABLA11 = SUBGRUPOS.groupby(["id", "dimension"], sort=False).p_igualdad.first().unstack()[list(GRUPOS)]
TABLA11.insert(0, "indicador", META.loc[TABLA11.index, "indicador"])
print(SUBGRUPOS.pivot_table(index="indicador", columns=["dimension", "grupo"], values="coef", sort=False).round(1).to_string())
print("\nTabla 11. Prueba de igualdad (p):"); print(TABLA11.round(3).to_string())
sig_g = SUBGRUPOS.groupby(["id", "dimension"]).p_igualdad.first()
print(f"\nDiferencias significativas: {int((sig_g < 0.05).sum())} de {len(sig_g)} pruebas")

In [ ]:
# Figura 5: cambio por grupo en los indicadores clave
orden = sum((niv for _, niv in GRUPOS.values()), [])
pos = [len(orden) - 1 - j + (0 if j < 2 else -0.5 if j < 5 else -1) for j in range(len(orden))]
color_dim = {"Sexo": BLUE, "Edad": CYAN, "Nivel socioeconómico": GREEN}
etiqueta_dim = {"Sexo": "sexo", "Edad": "edad", "Nivel socioeconómico": "NSE"}
fig, axs = plt.subplots(2, 4, figsize=(6.9, 5.6), dpi=150, sharey=True)
for ax, i in zip(axs.flat, INDICADORES_CLAVE):
    d = SUBGRUPOS[SUBGRUPOS.id == i].set_index("grupo").loc[orden]
    for yy, (_, r) in zip(pos, d.iterrows()):
        ax.plot([r.ic95_inf, r.ic95_sup], [yy, yy], color=color_dim[r.dimension], lw=1.3)
        ax.plot(r.coef, yy, "o", color=color_dim[r.dimension], ms=3.6)
    ax.axvline(0, color="#64748b", lw=.7)
    ax.set_title("\n".join(textwrap.wrap(META.loc[i, "indicador"], 26)), fontsize=6.6, color=NAVY, fontweight="bold", pad=11)
    difiere = [etiqueta_dim[dm] for dm, p in d.groupby("dimension").p_igualdad.first().items() if p < 0.05]
    if difiere:
        ax.text(.5, 1.005, "Difiere por " + " y ".join(difiere) + " (p < 0,05)", transform=ax.transAxes, ha="center", va="bottom", fontsize=5.4, color="#b4401f")
    ax.tick_params(labelsize=6); [ax.spines[s].set_visible(False) for s in ("top", "right")]; ax.grid(axis="x", color="#eef2f7", lw=.5)
for ax in axs[:, 0]:
    ax.set_yticks(pos); ax.set_yticklabels(orden, fontsize=6.2)
fig.text(.55, .01, "Diferencia ajustada (pp), IC 95%", ha="center", fontsize=7)
plt.tight_layout(rect=(0, .03, 1, 1)); plt.savefig(f"{CARPETA}/figura5_grupos.png", dpi=220); plt.show()

## 10. Victimización reportada y registros administrativos (Recuadro 3)
Asociación, a nivel de persona, entre haber sido víctima de robo o hurto (VI_4, VI_6 o VI_7) y el número de robos registrados en un radio de 200 m de la vivienda en los 24 meses previos a la encuesta.

In [ ]:
col_24m = [c for c in df.columns if str(c).strip().startswith("24 meses")][0]
robos_24m = pd.to_numeric(df[col_24m], errors="coerce").values
victima = pd.concat([victimizacion(k) for k in ("VI_4", "VI_6", "VI_7")], axis=1).max(axis=1).values

def corr_ponderada(a, b, w):
    m = ~(np.isnan(a) | np.isnan(b)); a, b, w = a[m], b[m], w[m]
    ma, mb = np.average(a, weights=w), np.average(b, weights=w)
    return np.average((a - ma) * (b - mb), weights=w) / np.sqrt(np.average((a - ma) ** 2, weights=w) * np.average((b - mb) ** 2, weights=w))

def auc(y, puntaje):
    r = stats.rankdata(puntaje); n1, n0 = (y == 1).sum(), (y == 0).sum()
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

filas = []
for ola, nombre in ((2023, "2023-2024"), (2025, "2025-2026")):
    m = (df["AÑO"] == ola).values & ~np.isnan(robos_24m)
    filas.append({"ola": nombre, "correlacion": corr_ponderada(robos_24m[m], victima[m], W[m]), "AUC": auc(victima[m], robos_24m[m]),
                  "pct_victima": 100 * np.average(victima[m], weights=W[m])})
CONCORDANCIA = pd.DataFrame(filas)
CONCORDANCIA.round(3)

## 11. Exportar resultados
Genera un Excel con todas las tablas del informe y un `.zip` con las figuras. En Colab se descargan automáticamente.

In [ ]:
SALIDA = f"{CARPETA}/resultados_estudio_EVPI_DMQ.xlsx"
with pd.ExcelWriter(SALIDA, engine="openpyxl") as xl:
    RES.to_excel(xl, sheet_name="Indicadores_56", index=False)
    TABLA1.to_excel(xl, sheet_name="T1_seleccionados")
    TABLA_MUESTRA.to_excel(xl, sheet_name="T2_muestra_AZ")
    BALANCE.to_excel(xl, sheet_name="T3_balance", index=False)
    TABLA4.to_excel(xl, sheet_name="T4_bloques")
    for k, b in enumerate(BLOQUES, start=5):
        tabla_bloque(b).to_excel(xl, sheet_name=f"T{k}_bloque_{b[0]}", index=False)
    TABLA11.to_excel(xl, sheet_name="T11_igualdad_grupos")
    TABLA12.to_excel(xl, sheet_name="T12_robustez")
    VARIANTES.to_excel(xl, sheet_name="Variantes_detalle", index=False)
    SUBGRUPOS.to_excel(xl, sheet_name="Grupos_detalle", index=False)
    CONCORDANCIA.to_excel(xl, sheet_name="Recuadro3_concordancia", index=False)
    META.reset_index().to_excel(xl, sheet_name="Anexo_A_definiciones", index=False)

ZIP = f"{CARPETA}/figuras_estudio.zip"
with zipfile.ZipFile(ZIP, "w") as z:
    for f in sorted(os.listdir(CARPETA)):
        if f.endswith(".png"):
            z.write(f"{CARPETA}/{f}", f)
print("Guardado:", SALIDA, "y", ZIP)
try:
    from google.colab import files
    files.download(SALIDA); files.download(ZIP)
except ImportError:
    pass